In [ ]:
#Backpropagation
import numpy as np
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.weights_input_hidden = np.random.randn(
            self.input_size, self.hidden_size)
        self.weights_hidden_output = np.random.randn(
            self.hidden_size, self.output_size)

        self.bias_hidden = np.zeros((1, self.hidden_size))
        self.bias_output = np.zeros((1, self.output_size))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        return x * (1 - x)

    def feedforward(self, X):
        self.hidden_activation = np.dot(
            X, self.weights_input_hidden) + self.bias_hidden
        self.hidden_output = self.sigmoid(self.hidden_activation)

        self.output_activation = np.dot(
            self.hidden_output, self.weights_hidden_output) + self.bias_output
        self.predicted_output = self.sigmoid(self.output_activation)

        return self.predicted_output

    def backward(self, X, y, learning_rate):
        output_error = y - self.predicted_output
        output_delta = output_error * \
            self.sigmoid_derivative(self.predicted_output)

        hidden_error = np.dot(output_delta, self.weights_hidden_output.T)
        hidden_delta = hidden_error * self.sigmoid_derivative(self.hidden_output)

        self.weights_hidden_output += np.dot(self.hidden_output.T,
                                             output_delta) * learning_rate
        self.bias_output += np.sum(output_delta, axis=0,
                                   keepdims=True) * learning_rate
        self.weights_input_hidden += np.dot(X.T, hidden_delta) * learning_rate
        self.bias_hidden += np.sum(hidden_delta, axis=0,
                                   keepdims=True) * learning_rate
    def train(self, X, y, epochs, learning_rate):
        for epoch in range(epochs):
            output = self.feedforward(X)
            self.backward(X, y, learning_rate)
            if epoch % 4000 == 0:
                loss = np.mean(np.square(y - output))
                print(f"Epoch {epoch}, Loss:{loss}")

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

nn = NeuralNetwork(input_size=2, hidden_size=4, output_size=1)
nn.train(X, y, epochs=10000, learning_rate=0.1)

output = nn.feedforward(X)
print("Predictions after training:")
print(output)

Epoch 0, Loss:0.28611091540347944
Epoch 4000, Loss:0.007986408233721123
Epoch 8000, Loss:0.00209435184892058
Predictions after training:
[[0.02853891]
 [0.96496512]
 [0.95889171]
 [0.0471844 ]]


### Giải thích mã nguồn Backpropagation và Nhận xét kết quả

#### 1. Giải thích cấu trúc và hoạt động của Mạng Nơ-ron (Neural Network)
* **Khởi tạo (`__init__`):**
  * Mạng được thiết kế gồm 3 lớp: Lớp đầu vào (Input layer - 2 node), Lớp ẩn (Hidden layer - 4 node) và Lớp đầu ra (Output layer - 1 node).
  * Trọng số (`weights`) được khởi tạo ngẫu nhiên theo phân phối chuẩn, trong khi bias được khởi tạo bằng 0.
* **Hàm kích hoạt Sigmoid:**
  * `sigmoid(x)`: Ánh xạ các giá trị đầu vào về khoảng $(0, 1)$, giúp mô hình học các quan hệ phi tuyến.
  * `sigmoid_derivative(x)`: Đạo hàm của hàm sigmoid ($y \cdot (1 - y)$), được sử dụng trong quá trình lan truyền ngược để tính toán gradient.
* **Lan truyền xuôi (`feedforward`):**
  * Tính toán giá trị kích hoạt tại lớp ẩn từ dữ liệu đầu vào và cập nhật qua hàm sigmoid.
  * Tiếp tục tính toán từ lớp ẩn sang lớp đầu ra để đưa ra dự đoán cuối cùng (`predicted_output`).
* **Lan truyền ngược (`backward`):**
  * Đo lường sai số giữa thực tế ($y$) và dự đoán.
  * Tính toán các mức thay đổi (`delta`) dựa trên đạo hàm của hàm sigmoid để xác định mức độ ảnh hưởng của từng trọng số tới sai số.
  * Cập nhật trọng số và bias theo hướng giảm thiểu sai số với hệ số học tập (`learning_rate`).
* **Quá trình huấn luyện (`train`):**
  * Lặp lại chu trình Lan truyền xuôi -> Lan truyền ngược trong 10,000 chu kỳ (`epochs`).

#### 2. Nhận xét kết quả thực nghiệm
* **Quá trình giảm lỗi (Loss):**
  * **Epoch 0:** Loss đạt mức khá cao (~0.286) do trọng số ban đầu được khởi tạo ngẫu nhiên.
  * **Epoch 4000:** Loss giảm mạnh xuống còn ~0.0079.
  * **Epoch 8000:** Loss chỉ còn ~0.0020.
  * Sự sụt giảm liên tục của hàm lỗi chứng minh thuật toán lan truyền ngược đang hoạt động cực kỳ hiệu quả, các trọng số đang được tối ưu hóa tốt.

* **Độ chính xác của dự đoán:**
  * Đang giải quyết bài toán cổng logic **XOR**:
    * Đầu vào `[0, 0]` $\rightarrow$ Thực tế `[0]` $\rightarrow$ Dự đoán đạt `0.028` (Rất gần 0).
    * Đầu vào `[0, 1]` $\rightarrow$ Thực tế `[1]` $\rightarrow$ Dự đoán đạt `0.964` (Rất gần 1).
    * Đầu vào `[1, 0]` $\rightarrow$ Thực tế `[1]` $\rightarrow$ Dự đoán đạt `0.958` (Rất gần 1).
    * Đầu vào `[1, 1]` $\rightarrow$ Thực tế `[0]` $\rightarrow$ Dự đoán đạt `0.047` (Rất gần 0).
  * **Kết luận:** Mô hình đã hội tụ thành công và học được phân tách phi tuyến tính của hàm XOR một cách chính xác.